<a href="https://colab.research.google.com/github/WalterVillatoro1204/marketing-analytics-walter-villatoro/blob/main/Semana_3/breakout_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Breakout Room #2 — Calcular el Churn

**Objetivo:**
- Detectar a los clientes que están en riesgo de perderse.
- ¿Lograremos retenerlos? (la vamos a contestar en la clase, con los números que traigan de aquí)

Esta actividad aplica directo la slide que acaban de ver: **las dos ventanas**. Van a construir un
modelo de churn de verdad, con un corte explícito entre lo que el modelo puede ver (ventana de
observación) y lo que está tratando de adivinar (ventana objetivo). Donde dice **`# TODO`** hay
algo para que ustedes completen.


In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# URLs directas de tu repositorio de GitHub
base_url = "https://raw.githubusercontent.com/WalterVillatoro1204/marketing-analytics-walter-villatoro/main/Semana_3/data"

transactions = pd.read_csv(f"{base_url}/transactions.csv", parse_dates=["transaction_date"])
customers = pd.read_csv(f"{base_url}/customers.csv", parse_dates=["signup_date", "last_active_date"])

print(f"{len(customers):,} clientes · {len(transactions):,} transacciones")
print(f"Rango de fechas: {transactions['transaction_date'].min().date()} a {transactions['transaction_date'].max().date()}")

2,000 clientes · 164,239 transacciones
Rango de fechas: 2024-09-02 a 2026-08-30


## Paso 1

Definición de fechas de corte

- **Ventana de observación** = todo lo que pasó HASTA el corte. De aquí sale todo lo que el modelo
  puede usar.
- **Ventana objetivo** = los 90 días DESPUÉS del corte. De aquí sale la etiqueta (¿volvió o no?).


In [10]:
MAX_DATE = transactions["transaction_date"].max()
CUTOFF = MAX_DATE - pd.Timedelta(days=90)

observacion = transactions[transactions["transaction_date"] <= CUTOFF]
objetivo = transactions[
    (transactions["transaction_date"] > CUTOFF) & (transactions["transaction_date"] <= MAX_DATE)
]

print(f"Corte: {CUTOFF.date()}")
print(f"Ventana de observación: hasta {CUTOFF.date()}  →  {observacion['customer_id'].nunique():,} clientes activos")
print(f"Ventana objetivo: {CUTOFF.date()} a {MAX_DATE.date()}  →  {objetivo['customer_id'].nunique():,} clientes activos")


Corte: 2026-06-01
Ventana de observación: hasta 2026-06-01  →  1,809 clientes activos
Ventana objetivo: 2026-06-01 a 2026-08-30  →  1,017 clientes activos


## Paso 2 — construyan las features (solo con la ventana de observación)

Trabajamos únicamente con clientes "establecidos": que ya existían (se registraron) y que ya
habían comprado al menos una vez antes del corte. Un cliente nuevo que se registró ayer no tiene
historia suficiente para que RFM diga algo sensato de él.


In [11]:
establecidos = customers[customers["signup_date"] <= CUTOFF]["customer_id"]
obs_establecidos = observacion[observacion["customer_id"].isin(establecidos)]

print(f"Clientes establecidos al corte: {len(establecidos):,}")
print(f"...de los cuales compraron al menos una vez antes del corte: {obs_establecidos['customer_id'].nunique():,}")


Clientes establecidos al corte: 1,825
...de los cuales compraron al menos una vez antes del corte: 1,809


In [12]:
# TODO: usando el datafram "obs_establecidos" calculen recency_days para cada cliente — días entre su ÚLTIMA compra ANTES DEL CORTE y el corte.
# Pista: agrupen obs_establecidos por customer_id,obs_establecidos.groupby("customer_id"), tomen la fecha máxima de transaction_date,
# y réstenla de CUTOFF (CUTOFF - fecha_max).dt.days

# El resultado final es un Dataframe llamado "features" con las columnas ["customer_id", "recency_days", "frequency", "monetary", "signup_date", "tenure_days"]
# ejemplo (no es codigo literal):
# recency_days = (CUTOFF - transaction_date.max)
# frequency = (transaction_date.count)
# monetary = (amount.sum)
# tenure_days = (CUTOFF-signup_date)

# 1. Agrupar transacciones de observación para calcular recency, frequency y monetary
rfm = obs_establecidos.groupby("customer_id").agg(
    last_transaction=("transaction_date", "max"),
    frequency=("transaction_date", "count"),
    monetary=("amount", "sum")
).reset_index()

# 2. Calcular los días transcurridos desde la última compra hasta el corte
rfm["recency_days"] = (CUTOFF - rfm["last_transaction"]).dt.days

# 3. Unir con la tabla customers para obtener la fecha de registro
features = rfm.merge(customers[["customer_id", "signup_date"]], on="customer_id", how="inner")

# 4. Calcular los días de antigüedad desde el registro hasta el corte
features["tenure_days"] = (CUTOFF - features["signup_date"]).dt.days

# 5. Seleccionar exactamente las columnas solicitadas
features = features[["customer_id", "recency_days", "frequency", "monetary", "signup_date", "tenure_days"]]

# Mostrar las primeras filas
features.head()



,customer_id,recency_days,frequency,monetary,signup_date,tenure_days
0,1,475,1,16.00,2025-02-04,482
1,2,112,22,161.55,2025-10-12,232
2,3,5,3,31.77,2026-03-12,81
3,4,0,56,593.97,2025-04-21,406
4,5,0,252,1876.32,2025-02-13,473


## Paso 3 — construyan la etiqueta (solo con la ventana objetivo)

`churned = 1` si el cliente NO compró en los 90 días después del corte. `churned = 0` si sí volvió.


In [14]:
# TODO: construyan el set de clientes que SÍ compraron en la ventana objetivo,
# y con eso la columna "churned" (1 = no volvió, 0 = sí volvió).
# Pista: objetivo["customer_id"].unique() les da quién compró en la ventana objetivo.

compraron_despues = set(objetivo["customer_id"].unique())
features["churned"] = features["customer_id"].isin(compraron_despues).astype(int)

print("Distribucion de churned:")
print(features["churned"].value_counts(normalize=True))
print(features.head())

Distribucion de churned:
churned
0    0.543394
1    0.456606
Name: proportion, dtype: float64
   customer_id  recency_days  frequency  monetary signup_date  tenure_days  \
0            1           475          1     16.00  2025-02-04          482   
1            2           112         22    161.55  2025-10-12          232   
2            3             5          3     31.77  2026-03-12           81   
3            4             0         56    593.97  2025-04-21          406   
4            5             0        252   1876.32  2025-02-13          473   

   churned  
0        0  
1        0  
2        1  
3        1  
4        1  


## Paso 4 — entrenen el modelo

Una regresión logística simple con cuatro variables, todas calculadas SOLO con datos de antes del
corte.


In [ ]:
X = features[["recency_days", "frequency", "monetary", "tenure_days"]]
y = features["churned"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

proba_test = modelo.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba_test)
print(f"AUC del modelo (correcto, sin fuga): {auc:.3f}")

pd.Series(modelo.coef_[0], index=X.columns).rename("coeficiente")


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

# 1) Por qué el accuracy engaña
tasa_base = y_test.mean()
acc_modelo = accuracy_score(y_test, (proba_test >= 0.5).astype(int))
print(f"Tasa base de churn:                {tasa_base:.1%}")
print(f"Accuracy del modelo:               {acc_modelo:.3f}")
print(f"Accuracy de 'no se va nadie':      {1 - tasa_base:.3f}")
print()

# 2) Precision@k, Recall@k y Lift@k
y_arr = np.asarray(y_test)
orden = np.argsort(proba_test)[::-1]          # de mayor a menor riesgo

filas = []
for pct in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    k = int(len(y_arr) * pct)
    y_top = y_arr[orden[:k]]
    precision_k = y_top.mean()
    recall_k = y_top.sum() / y_arr.sum()
    filas.append({
        "% contactado": f"{pct:.0%}",
        "k": k,
        "Precision@k": round(precision_k, 3),
        "Recall@k": round(recall_k, 3),
        "Lift@k": round(precision_k / tasa_base, 2),
    })

pd.DataFrame(filas)

## Paso 5 — Usamos (erroneamente) trasnactions que tiene todas las transacciones

Esta celda ya está resuelta — no hace falta que la editen, solo que la corran y la comparen con
su AUC del Paso 4. Aquí calculamos la recencia MAL: usando TODAS las transacciones (incluyendo las
de la ventana objetivo), medida desde el final de los datos en vez de desde el corte. Es
exactamente el error de la slide anterior.


In [ ]:
todas_tx_establecidos = transactions[transactions["customer_id"].isin(establecidos)]
ultima_compra_cualquiera = todas_tx_establecidos.groupby("customer_id")["transaction_date"].max()

features_fuga = features.merge(
    ultima_compra_cualquiera.rename("ultima_compra_cualquiera"), on="customer_id"
)
features_fuga["recency_leaky"] = (MAX_DATE - features_fuga["ultima_compra_cualquiera"]).dt.days

X_leaky = features_fuga[["recency_leaky", "frequency", "monetary", "tenure_days"]]
y_leaky = features_fuga["churned"]

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y_leaky, test_size=0.3, random_state=42, stratify=y_leaky
)
modelo_fuga = LogisticRegression(max_iter=1000)
modelo_fuga.fit(Xl_train, yl_train)
auc_fuga = roc_auc_score(yl_test, modelo_fuga.predict_proba(Xl_test)[:, 1])

print(f"AUC del modelo CON fuga: {auc_fuga:.3f}")
print(f"AUC del modelo correcto (Paso 4): {auc:.3f}")
print()
print("Un AUC así de perfecto no es un buen modelo: es la señal de que algo del futuro se coló al pasado.")


## Paso 6 — la gráfica: curva ROC, correcto vs. con fuga


In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba_test)
fpr_leaky, tpr_leaky, _ = roc_curve(yl_test, modelo_fuga.predict_proba(Xl_test)[:, 1])

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(fpr, tpr, color="#00704A", linewidth=3, label=f"Correcto (AUC={auc:.3f})")
ax.plot(fpr_leaky, tpr_leaky, color="#C4512F", linewidth=3, linestyle="--", label=f"Con fuga (AUC={auc_fuga:.3f})")
ax.plot([0, 1], [0, 1], color="#9A988E", linewidth=1.5, linestyle=":", label="Azar")
ax.set_xlabel("Falsos positivos")
ax.set_ylabel("Verdaderos positivos")
ax.set_title("Curva ROC: el modelo correcto vs. el que hace trampa")
ax.legend()
plt.tight_layout()
plt.savefig("roc_correcto_vs_fuga.png", dpi=150)
plt.show()


## Paso 7 — ¿a quién detectamos en riesgo?

Usamos el modelo CORRECTO (Paso 4) para puntuar a todo el set de prueba, y vemos quiénes son los
clientes de mayor riesgo.


In [ ]:
resultado = X_test.copy()
resultado["customer_id"] = features.loc[X_test.index, "customer_id"].values
resultado["prob_churn"] = proba_test # probabilidad en regresión logistica
resultado["churned_real"] = y_test.values

en_riesgo = resultado.sort_values("prob_churn", ascending=False).head(15)
en_riesgo[["customer_id", "recency_days", "frequency", "monetary", "prob_churn", "churned_real"]]


In [ ]:
# ¿Cuántos clientes de cada 10 que el modelo marca como "alto riesgo" (prob > 0.7) de verdad no volvieron?
alto_riesgo = resultado[resultado["prob_churn"] > 0.7]
print(f"Clientes marcados como alto riesgo (prob > 70%): {len(alto_riesgo)}")
print(f"De esos, realmente no volvieron: {alto_riesgo['churned_real'].mean():.1%}")


## Para la clase

1. ¿Qué AUC les dio el modelo correcto? ¿Y el que hace trampa?
2. En una frase: ¿por qué un AUC de 1.0 (o casi) es una mala noticia y no una buena?
3. Anoten cuántos clientes marcaron como "alto riesgo" (prob > 70%). Vamos a preguntar:
   **"¿a quién contactamos?"** — con esta lista en la mano, ¿cuál es su respuesta?
